# 🎧 อักษราลัย — ทดลองสร้าง MP3 ภาษาไทยฟรี

โน้ตบุ๊กนี้ใช้ [VachanaTTS](https://huggingface.co/VIZINTZOR/VachanaTTS) โมเดลภาษาไทยที่ระบุสัญญาอนุญาต MIT บน Hugging Face และรันใน Google Colab (ฟรีตามโควตาที่ Google จัดสรร) ไม่ใช้ API Key หรือคิดเงินเป็นรายตัวอักษร

**ขั้นตอน:** เลือก Runtime → Run all / เรียกใช้ทั้งหมด → เลือกเสียง → ทดลองข้อความสั้น → ถ้าต้องการทั้งตอนให้อัปโหลดไฟล์ `.txt` ในเซลล์ 3 → รอระบบสร้าง MP3 → ดาวน์โหลดไฟล์ไปอัปโหลดในอักษราลัยหลังเปิด R2 แล้ว

⚠️ ต้องตรวจสอบคุณภาพเสียงและสิทธิ์ใช้งานองค์ประกอบทั้งหมดก่อนนำไปใช้เชิงพาณิชย์จริง เนื้อหาที่อัปโหลดอยู่ใน Colab runtime ของคุณ ควรใช้เฉพาะข้อความที่คุณมีสิทธิ์

In [ ]:
!pip -q install vachanatts==0.0.7
!apt-get -qq update >/dev/null && apt-get -qq install -y ffmpeg >/dev/null
print('✅ ติดตั้งเสร็จแล้ว')

## 1. ตั้งค่าและทดลองเสียงสั้น

เปลี่ยนข้อความและรหัสเสียงได้: `th_f_1`, `th_f_2`, `th_m_1`, `th_m_2`

In [ ]:
from vachanatts import TTS
from IPython.display import Audio, display
SPEAKER = 'th_f_1' #@param ['th_f_1', 'th_f_2', 'th_m_1', 'th_m_2']
TEST_TEXT = 'พรานสิงห์เดินช้า ๆ ผ่านป่าทึบ เขาหยุดฟังเสียงกระซิบจากความมืด' #@param {type:'string'}
TTS(TEST_TEXT, voice=SPEAKER, output='aksaralai-preview.wav', volume=1.0, speed=1.0)
display(Audio(filename='aksaralai-preview.wav'))
print('✅ ทดลองเสียงแล้ว — หากพอใจค่อยสร้างเสียงทั้งตอนในขั้นถัดไป')

## 2. เลือกไฟล์นิยาย `.txt` ที่เป็น UTF-8

ไฟล์จะถูกแบ่งเป็นช่วงตามประโยคและวรรค เพื่อไม่ให้ส่งข้อความยาวเกินไปในคำสั่งเดียว มีช่วงเงียบเล็กน้อยระหว่างย่อหน้า และยังสามารถเปลี่ยนเสียงได้ในเซลล์ก่อนหน้า

In [ ]:
from google.colab import files
uploaded = files.upload()
choices = [n for n in uploaded if n.lower().endswith('.txt')]
if not choices:
    raise ValueError('กรุณาเลือกไฟล์ .txt ก่อน')
source_name = choices[0]
novel_text = uploaded[source_name].decode('utf-8-sig')
if not novel_text.strip():
    raise ValueError('ไฟล์ไม่มีเนื้อหา')
print('📚 รับไฟล์:',source_name,'จำนวน',len(novel_text),'ตัวอักษร')

## 3. สร้างและดาวน์โหลด MP3

เริ่มจากตอนสั้น ๆ ก่อนเพื่อทดสอบการออกเสียง ระหว่างประมวลผลห้ามปิด Colab หน้าเว็บ โปรแกรมอาจช้าหรือหยุดหากสิทธิ์ประมวลผลฟรีหมด

In [ ]:
import re, subprocess
from pathlib import Path
OUTPUT = 'aksaralai-chapter.mp3'
MAX_CHARACTERS = 130

def sections(text, max_chars=MAX_CHARACTERS):
    from pythainlp.tokenize import word_tokenize
    out=[]
    for paragraph in re.split(r'\n+', text):
        paragraph=paragraph.strip()
        if not paragraph:
            continue
        tokens=word_tokenize(paragraph,engine='newmm',keep_whitespace=True)
        buffer=''
        for token in tokens:
            if buffer and len(buffer)+len(token)>max_chars:
                out.append(buffer.strip())
                buffer=''
            buffer+=token
        if buffer.strip():out.append(buffer.strip())
    return [x for x in out if x]

chunks=sections(novel_text)
print('จำนวนช่วงเสียง:',len(chunks))
if not chunks:raise ValueError('ไม่มีข้อความให้อ่าน')
if len(chunks)>2500:raise ValueError('ข้อความยาวมาก แนะนำแบ่งเป็นหลายไฟล์')
folder=Path('aksaralai_segments')
folder.mkdir(exist_ok=True)
wavs=[]
for number,txt in enumerate(chunks,1):
    path=folder/f'{number:05d}.wav'
    TTS(txt,voice=SPEAKER,output=str(path),volume=1.0,speed=1.0)
    wavs.append(path)
    if number%20==0 or number==len(chunks):
        print('สร้างเสียงแล้ว',number,'/',len(chunks))

# Join equal-format WAV segments and compress once, without repeated MP3 re-encoding.
# All paths are generated locally, never taken from the uploaded filename.
manifest=folder/'concat.txt'
with manifest.open('w',encoding='utf8') as fp:
    for wav_path in wavs:
        fp.write("file '"+wav_path.resolve().as_posix()+"'\n")
subprocess.run(['ffmpeg','-hide_banner','-loglevel','error','-y','-f','concat','-safe','0','-i',str(manifest),'-ac','1','-b:a','64k',OUTPUT],check=True)
size=Path(OUTPUT).stat().st_size
print(f'สร้าง MP3 สำเร็จ: {size/1048576:.1f} MB')
if size>40*1024*1024:
    print('⚠️ ไฟล์เกิน 40 MB ที่อักษราลัยรองรับ กรุณาแบ่งตอนก่อน')
files.download(OUTPUT)